In [2]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 2.6 MB/s eta 0:00:00


In [4]:
import redis

print("Redis library imported successfully!")

Redis library imported successfully!


In [5]:
REDIS_HOST = "129.153.75.221"
REDIS_PORT = 6379
REDIS_USERNAME = "default"

# No password was provided
REDIS_PASSWORD = None

print("Redis configuration loaded!")
print("Host:", REDIS_HOST)
print("Port:", REDIS_PORT)
print("Username:", REDIS_USERNAME)

Redis configuration loaded!
Host: 129.153.75.221
Port: 6379
Username: default


In [6]:
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    username=REDIS_USERNAME,
    password=REDIS_PASSWORD,
    decode_responses=True,
    socket_connect_timeout=10,
    socket_timeout=10
)

print("Redis client created successfully!")

Redis client created successfully!


In [7]:
try:
    result = redis_client.ping()

    if result:
        print("Connected to Redis successfully!")
        print("Redis PING:", result)

except Exception as e:
    print("Redis connection failed!")
    print("Error:", e)

Redis connection failed!
Error: invalid username-password pair


In [8]:
from getpass import getpass

REDIS_PASSWORD = getpass("Enter Redis password: ")

Enter Redis password: ··········


In [9]:
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    username=REDIS_USERNAME,
    password=REDIS_PASSWORD,
    decode_responses=True,
    socket_connect_timeout=10,
    socket_timeout=10
)

print("Redis client created successfully!")

Redis client created successfully!


In [10]:
try:
    result = redis_client.ping()

    if result:
        print("Connected to Redis successfully!")
        print("Redis PING:", result)

except Exception as e:
    print("Redis connection failed!")
    print("Error:", e)

Connected to Redis successfully!
Redis PING: True


In [18]:
import json

print("JSON library imported successfully!")

JSON library imported successfully!


In [19]:
# ============================================
# PHASE 4 - REDIS SET / WRITE TEST
# ============================================

cache_key = "student_wellness:1001"

cache_data = {
    "student_id": 1001,
    "predicted_risk": "Medium",
    "probabilities": {
        "Low": 0.23,
        "Medium": 70.53,
        "High": 29.25
    }
}

# Convert dictionary to JSON string before storing
redis_client.set(
    cache_key,
    json.dumps(cache_data)
)

print("Data stored in Redis successfully!")
print("Key:", cache_key)
print("Data:", cache_data)

Data stored in Redis successfully!
Key: student_wellness:1001
Data: {'student_id': 1001, 'predicted_risk': 'Medium', 'probabilities': {'Low': 0.23, 'Medium': 70.53, 'High': 29.25}}


In [20]:
# ============================================
# PHASE 4 - REDIS GET / READ TEST
# ============================================

cached_value = redis_client.get(cache_key)

if cached_value:
    cached_data = json.loads(cached_value)

    print("Data retrieved from Redis successfully!")
    print("Key:", cache_key)
    print("Cached Data:", cached_data)

else:
    print("No cached data found.")

Data retrieved from Redis successfully!
Key: student_wellness:1001
Cached Data: {'student_id': 1001, 'predicted_risk': 'Medium', 'probabilities': {'Low': 0.23, 'Medium': 70.53, 'High': 29.25}}


In [21]:
# ============================================
# PHASE 4 - REDIS UPDATE TEST
# ============================================

updated_cache_data = {
    "student_id": 1001,
    "predicted_risk": "High",
    "probabilities": {
        "Low": 0.10,
        "Medium": 20.40,
        "High": 79.50
    }
}

redis_client.set(
    cache_key,
    json.dumps(updated_cache_data)
)

print("Cached data updated successfully!")
print("Updated Data:", updated_cache_data)

Cached data updated successfully!
Updated Data: {'student_id': 1001, 'predicted_risk': 'High', 'probabilities': {'Low': 0.1, 'Medium': 20.4, 'High': 79.5}}


In [22]:
# Verify updated data

updated_value = redis_client.get(cache_key)

if updated_value:
    print("Data currently stored in Redis:")
    print(json.loads(updated_value))

Data currently stored in Redis:
{'student_id': 1001, 'predicted_risk': 'High', 'probabilities': {'Low': 0.1, 'Medium': 20.4, 'High': 79.5}}


In [23]:
# ============================================
# PHASE 4 - REDIS DELETE TEST
# ============================================

deleted_count = redis_client.delete(cache_key)

if deleted_count == 1:
    print("Cached data deleted successfully!")
else:
    print("Key was not found.")

Cached data deleted successfully!


In [24]:
# Verify deletion

remaining_data = redis_client.get(cache_key)

if remaining_data is None:
    print("Verified: Data no longer exists in Redis.")
else:
    print("Data still exists:", remaining_data)

Verified: Data no longer exists in Redis.


In [26]:
ttl_key = "student_wellness:test"

ttl_data = {
    "student_id": 1001,
    "predicted_risk": "Medium"
}

redis_client.set(
    ttl_key,
    json.dumps(ttl_data),
    ex=300
)

print("Temporary cache created!")
print("Key:", ttl_key)
print("TTL:", redis_client.ttl(ttl_key), "seconds")

Temporary cache created!
Key: student_wellness:test
TTL: 300 seconds


In [30]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 23 - Cache HIT Test
# ============================================

import requests
import json

# Same data used in the previous successful Cache MISS test
test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

# Send the exact same request again
response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ad3eb2dcb90>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [32]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 24 - Recreate Flask Application
# ============================================

from flask import Flask, request, jsonify

app = Flask(__name__)

print("Flask application recreated successfully!")

Flask application recreated successfully!


In [33]:
# ============================================
# Cell 25 - Home Route
# ============================================

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Predictive Campus Life Wellness Sentinel - Phase 5",
        "status": "API is running",
        "endpoint": "/predict"
    })

print("Home route registered successfully!")

Home route registered successfully!


In [34]:
print("Model loaded:", "model" in globals())
print("Redis connected:", "redis_client" in globals())
print("RabbitMQ publisher:", "publish_prediction_to_rabbitmq" in globals())

Model loaded: False
Redis connected: True
RabbitMQ publisher: False


In [37]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 26A - Install Required Packages
# ============================================

!pip install pika redis flask joblib pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 1.3 MB/s eta 0:00:00


In [40]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 26A - Upload Pretrained Model
# ============================================

from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Saving wellness_randomforest_pipeline.pkl to wellness_randomforest_pipeline.pkl
Uploaded files:
- wellness_randomforest_pipeline.pkl


In [41]:
import os

print("Model file exists:", os.path.exists("wellness_randomforest_pipeline.pkl"))

Model file exists: True


In [42]:
# ============================================
# Cell 26B - Load Pretrained Random Forest
# ============================================

import joblib
import pandas as pd
import json
import pika

MODEL_PATH = "wellness_randomforest_pipeline.pkl"

model_package = joblib.load(MODEL_PATH)

model = model_package["model"]
FEATURE_COLUMNS = model_package["features"]
RISK_MAPPING = model_package["risk_mapping"]

print("Pretrained Random Forest model loaded successfully!")
print("Number of ML features:", len(FEATURE_COLUMNS))
print("Risk mapping:", RISK_MAPPING)

Pretrained Random Forest model loaded successfully!
Number of ML features: 14
Risk mapping: {0: 'Low', 1: 'Medium', 2: 'High'}


In [43]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 27 - Reliable RabbitMQ Publisher
# ============================================

def publish_prediction_to_rabbitmq(prediction_data):

    connection = None

    try:
        credentials = pika.PlainCredentials(
            RABBITMQ_USERNAME,
            RABBITMQ_PASSWORD
        )

        parameters = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            heartbeat=60,
            blocked_connection_timeout=300
        )

        # Create a fresh connection for each publish
        connection = pika.BlockingConnection(parameters)

        channel = connection.channel()

        channel.queue_declare(
            queue=QUEUE_NAME,
            durable=True
        )

        channel.confirm_delivery()

        message = json.dumps(prediction_data)

        channel.basic_publish(
            exchange="",
            routing_key=QUEUE_NAME,
            body=message,
            properties=pika.BasicProperties(
                delivery_mode=2,
                content_type="application/json"
            )
        )

        print("Prediction event published to RabbitMQ")

        return True

    except Exception as e:

        print("RabbitMQ publish failed:", str(e))

        return False

    finally:

        if connection is not None and connection.is_open:
            connection.close()

In [44]:
# ============================================
# Cell 28 - Verify Integration Components
# ============================================

print("Model loaded:", "model" in globals())
print("ML features:", len(FEATURE_COLUMNS))
print("Redis connected:", "redis_client" in globals())
print(
    "RabbitMQ publisher:",
    "publish_prediction_to_rabbitmq" in globals()
)

Model loaded: True
ML features: 14
Redis connected: True
RabbitMQ publisher: True


In [45]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 29 - Integrated /predict Endpoint
# ============================================

import hashlib


@app.route("/predict", methods=["POST"])
def predict():

    try:
        # --------------------------------------------
        # 1. Receive request through Flask
        # --------------------------------------------

        data = request.get_json()

        if not data:
            return jsonify({
                "error": "No JSON input provided"
            }), 400

        # --------------------------------------------
        # 2. Validate required ML features
        # --------------------------------------------

        missing_features = [
            feature
            for feature in FEATURE_COLUMNS
            if feature not in data
        ]

        if missing_features:
            return jsonify({
                "error": "Missing required features",
                "missing_features": missing_features
            }), 400

        # --------------------------------------------
        # 3. Create Redis cache key
        # --------------------------------------------

        cache_input = {
            feature: data[feature]
            for feature in FEATURE_COLUMNS
        }

        cache_string = json.dumps(
            cache_input,
            sort_keys=True
        )

        cache_hash = hashlib.md5(
            cache_string.encode()
        ).hexdigest()

        student_id = int(data.get("student_id", 0))

        cache_key = (
            f"student_wellness:"
            f"{student_id}:"
            f"{cache_hash}"
        )

        print("\n========================================")
        print("New prediction request")
        print("Student ID:", student_id)
        print("Cache Key:", cache_key)

        # --------------------------------------------
        # 4. Check Redis cache
        # --------------------------------------------

        cached_result = redis_client.get(cache_key)

        if cached_result:

            print("Redis Cache: HIT")
            print("Returning cached result")

            cached_response = json.loads(cached_result)

            cached_response["cache"] = {
                "hit": True,
                "key": cache_key
            }

            print("========================================\n")

            return jsonify(cached_response), 200

        # --------------------------------------------
        # Cache MISS
        # --------------------------------------------

        print("Redis Cache: MISS")
        print("Invoking Random Forest model...")

        # --------------------------------------------
        # 5. Invoke ML model
        # --------------------------------------------

        input_df = pd.DataFrame([data])

        input_df = input_df[FEATURE_COLUMNS]

        predicted_class = model.predict(input_df)[0]

        probabilities = model.predict_proba(input_df)[0]

        predicted_risk = RISK_MAPPING[predicted_class]

        probability_response = {
            "Low": round(float(probabilities[0]) * 100, 2),
            "Medium": round(float(probabilities[1]) * 100, 2),
            "High": round(float(probabilities[2]) * 100, 2)
        }

        print("Prediction:", predicted_risk)

        # --------------------------------------------
        # 6. Prepare prediction result
        # --------------------------------------------

        prediction_result = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response
        }

        # --------------------------------------------
        # 7. Store result in Redis
        # --------------------------------------------

        redis_client.set(
            cache_key,
            json.dumps(prediction_result),
            ex=300
        )

        print("Prediction stored in Redis")
        print("Cache TTL: 300 seconds")

        # --------------------------------------------
        # 8. Publish event to RabbitMQ
        # --------------------------------------------

        rabbitmq_message = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response
        }

        publish_status = publish_prediction_to_rabbitmq(
            rabbitmq_message
        )

        # --------------------------------------------
        # 9. Return final response
        # --------------------------------------------

        response = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response,
            "cache": {
                "hit": False,
                "stored": True,
                "ttl_seconds": 300,
                "key": cache_key
            },
            "rabbitmq": {
                "published": publish_status,
                "queue": QUEUE_NAME
            }
        }

        print("Flask response generated")
        print("========================================\n")

        return jsonify(response), 200

    except Exception as e:

        print("Prediction error:", str(e))

        return jsonify({
            "error": str(e)
        }), 500

In [46]:
# ============================================
# Cell 30 - Verify Flask Routes
# ============================================

print(app.url_map)

Map([<Rule '/static/<filename>' (GET, OPTIONS, HEAD) -> static>,
 <Rule '/' (GET, OPTIONS, HEAD) -> home>,
 <Rule '/predict' (OPTIONS, POST) -> predict>])


In [47]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 31 - Start Flask Server
# ============================================

import threading

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask,
    daemon=True
)

flask_thread.start()

print("Flask server started successfully!")
print("Server running on port 5000")


Flask server started successfully!
Server running on port 5000


In [48]:
# ============================================
# Cell 32 - Redis Cache HIT Test
# ============================================

import requests
import json

test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))


New prediction request
Student ID: 1001
Cache Key: student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1
Redis Cache: MISS
Invoking Random Forest model...


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:51:24] "POST /predict HTTP/1.1" 500 -


Prediction: Medium
Prediction stored in Redis
Cache TTL: 300 seconds
RabbitMQ publish failed: name 'RABBITMQ_USERNAME' is not defined
Prediction error: name 'QUEUE_NAME' is not defined
Status Code: 500
Response:
{
    "error": "name 'QUEUE_NAME' is not defined"
}


In [49]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 33 - Restore RabbitMQ Configuration
# ============================================

RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"
RABBITMQ_VHOST = "/"

QUEUE_NAME = "student_wellness_predictions"

print("RabbitMQ configuration restored!")
print("Host:", RABBITMQ_HOST)
print("Port:", RABBITMQ_PORT)
print("Username:", RABBITMQ_USERNAME)
print("Queue:", QUEUE_NAME)

RabbitMQ configuration restored!
Host: 129.153.75.221
Port: 5672
Username: bytesmart_interns
Queue: student_wellness_predictions


In [50]:
# ============================================
# Cell 34 - RabbitMQ Password
# ============================================

from getpass import getpass

RABBITMQ_PASSWORD = getpass(
    "Enter RabbitMQ password: "
)

print("RabbitMQ password loaded successfully!")

Enter RabbitMQ password: ··········
RabbitMQ password loaded successfully!


In [51]:
# ============================================
# Cell 35 - Verify RabbitMQ Configuration
# ============================================

print("RabbitMQ host configured:", bool(RABBITMQ_HOST))
print("RabbitMQ username configured:", bool(RABBITMQ_USERNAME))
print("RabbitMQ password configured:", bool(RABBITMQ_PASSWORD))
print("RabbitMQ queue configured:", bool(QUEUE_NAME))

RabbitMQ host configured: True
RabbitMQ username configured: True
RabbitMQ password configured: True
RabbitMQ queue configured: True


In [52]:
# ============================================
# Cell 36 - Test RabbitMQ Publisher
# ============================================

test_message = {
    "student_id": 9999,
    "predicted_risk": "Medium",
    "probabilities": {
        "Low": 10.0,
        "Medium": 70.0,
        "High": 20.0
    }
}

publish_status = publish_prediction_to_rabbitmq(
    test_message
)

print("Publish Status:", publish_status)

Prediction event published to RabbitMQ
Publish Status: True


In [53]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 37 - Clear Existing Test Cache
# ============================================

cache_key = "student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1"

deleted = redis_client.delete(cache_key)

print("Existing test cache deleted:", deleted)

Existing test cache deleted: 1


In [54]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 38 - Final Cache MISS Test
# ============================================

import requests
import json

test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))


New prediction request
Student ID: 1001
Cache Key: student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1
Redis Cache: MISS
Invoking Random Forest model...
Prediction: Medium
Prediction stored in Redis
Cache TTL: 300 seconds
Prediction event published to RabbitMQ


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:54:05] "POST /predict HTTP/1.1" 200 -


Flask response generated

Status Code: 200
Response:
{
    "cache": {
        "hit": false,
        "key": "student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1",
        "stored": true,
        "ttl_seconds": 300
    },
    "predicted_risk": "Medium",
    "probabilities": {
        "High": 29.25,
        "Low": 0.23,
        "Medium": 70.53
    },
    "rabbitmq": {
        "published": true,
        "queue": "student_wellness_predictions"
    },
    "student_id": 1001
}
